In [2]:
import warnings
import sys
import os
import time
import argparse
import nibabel as nib
from mpi4py import MPI
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import squareform, pdist
from sklearn.preprocessing import StandardScaler
from brainiak.searchlight.searchlight import Searchlight
import matplotlib.pyplot as plt
from nilearn import image, plotting, datasets
from ants import image_read, image_write, apply_transforms
import nilearn
from nilearn.image import resample_to_img
from os.path import join, exists
import ast

In [3]:
sl_dir = "./searchlight_rsa_results"
    
subs = ["01", "02", "03", "05", "06", "07", "08", "09", "10", "11", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31", "32"]

# define contrasts - ADDED congruent-auditory and congruent-visual for conjunction analysis
contrasts = ["congruent-auditory", "congruent-visual"] 

#"congruent-incongruent_mean", "congruent-unisensory_mean", "auditory-visual", "visual-auditory", 
#             "congruent-incongruent_A", "congruent-incongruent_V", "unisensory_mean-incongruent_mean",
#             "congruent-auditory", "congruent-visual"]  # NEW contrasts added

# loop through contrasts
for contrast_val in contrasts:
    
    # split conditions
    conds = contrast_val.split('-')
    
    # make contrast folder 
    contrast_dir = join(sl_dir, "contrasts", contrast_val) 
    os.makedirs(contrast_dir, exist_ok=True)

    # loop through subjects
    for sub in subs:
    
        # if congruent_mean
        if contrast_val == "congruent-incongruent_mean":
            
            # condition 1 path for curr. sub
            file_name_i = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_cond_i = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            cond_i_path = join(mni_folder_cond_i, file_name_i)

            # congruent A
            file_name_ii = f"sub-mm{sub}_cond-incongruent_A_searchlight-rsa_space_mni.nii.gz"
            mni_folder_cond_ii = f'{sl_dir}/incongruent_A/MNI152NLin2009cAsym'
            cond_ii_path = join(mni_folder_cond_ii, file_name_ii)
            
            # congruent V
            file_name_iii = f"sub-mm{sub}_cond-incongruent_V_searchlight-rsa_space_mni.nii.gz"
            mni_folder_cond_iii = f'{sl_dir}/incongruent_V/MNI152NLin2009cAsym'
            cond_iii_path = join(mni_folder_cond_iii, file_name_iii)

            # load files
            cond_i_mask = image_read(cond_i_path)
            cond_ii_mask = image_read(cond_ii_path)
            cond_iii_mask = image_read(cond_iii_path)
            
            # take mean across audio and visual incongruent masks
            mean_incongruent_mask = (cond_ii_mask + cond_iii_mask) / 2

            # compute contrast for current subject
            contrast_mask = cond_i_mask - mean_incongruent_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
            
        # congruent - mean(unisensory visual, unisensory audio) contrast
        elif contrast_val == "congruent-unisensory_mean":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)

            # unisensory audio condition path
            file_name_audio = f"sub-mm{sub}_cond-auditory_searchlight-rsa_space_mni.nii.gz"
            mni_folder_audio = f'{sl_dir}/auditory/MNI152NLin2009cAsym'
            audio_path = join(mni_folder_audio, file_name_audio)
            
            # unisensory visual condition path
            file_name_visual = f"sub-mm{sub}_cond-visual_searchlight-rsa_space_mni.nii.gz"
            mni_folder_visual = f'{sl_dir}/visual/MNI152NLin2009cAsym'
            visual_path = join(mni_folder_visual, file_name_visual)

            # load files
            congruent_mask = image_read(congruent_path)
            audio_mask = image_read(audio_path)
            visual_mask = image_read(visual_path)
            
            # take mean across audio and visual unisensory masks
            mean_unisensory_mask = (audio_mask + visual_mask) / 2

            # compute contrast for current subject
            contrast_mask = congruent_mask - mean_unisensory_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)

        # NEW: congruent - auditory contrast (for conjunction analysis)
        elif contrast_val == "congruent-auditory":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)

            # unisensory audio condition path
            file_name_audio = f"sub-mm{sub}_cond-auditory_searchlight-rsa_space_mni.nii.gz"
            mni_folder_audio = f'{sl_dir}/auditory/MNI152NLin2009cAsym'
            audio_path = join(mni_folder_audio, file_name_audio)

            # load files
            congruent_mask = image_read(congruent_path)
            audio_mask = image_read(audio_path)

            # compute contrast for current subject
            contrast_mask = congruent_mask - audio_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)

        # NEW: congruent - visual contrast (for conjunction analysis)
        elif contrast_val == "congruent-visual":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)

            # unisensory visual condition path
            file_name_visual = f"sub-mm{sub}_cond-visual_searchlight-rsa_space_mni.nii.gz"
            mni_folder_visual = f'{sl_dir}/visual/MNI152NLin2009cAsym'
            visual_path = join(mni_folder_visual, file_name_visual)

            # load files
            congruent_mask = image_read(congruent_path)
            visual_mask = image_read(visual_path)

            # compute contrast for current subject
            contrast_mask = congruent_mask - visual_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)

        # NEW: mean(unisensory) - mean(incongruent) contrast
        elif contrast_val == "unisensory_mean-incongruent_mean":
            
            # unisensory audio condition path
            file_name_audio = f"sub-mm{sub}_cond-auditory_searchlight-rsa_space_mni.nii.gz"
            mni_folder_audio = f'{sl_dir}/auditory/MNI152NLin2009cAsym'
            audio_path = join(mni_folder_audio, file_name_audio)
            
            # unisensory visual condition path
            file_name_visual = f"sub-mm{sub}_cond-visual_searchlight-rsa_space_mni.nii.gz"
            mni_folder_visual = f'{sl_dir}/visual/MNI152NLin2009cAsym'
            visual_path = join(mni_folder_visual, file_name_visual)

            # incongruent A condition path
            file_name_incong_A = f"sub-mm{sub}_cond-incongruent_A_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_A = f'{sl_dir}/incongruent_A/MNI152NLin2009cAsym'
            incong_A_path = join(mni_folder_incong_A, file_name_incong_A)
            
            # incongruent V condition path
            file_name_incong_V = f"sub-mm{sub}_cond-incongruent_V_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_V = f'{sl_dir}/incongruent_V/MNI152NLin2009cAsym'
            incong_V_path = join(mni_folder_incong_V, file_name_incong_V)

            # load files
            audio_mask = image_read(audio_path)
            visual_mask = image_read(visual_path)
            incong_A_mask = image_read(incong_A_path)
            incong_V_mask = image_read(incong_V_path)
            
            # take mean across audio and visual unisensory masks
            mean_unisensory_mask = (audio_mask + visual_mask) / 2
            
            # take mean across audio and visual incongruent masks
            mean_incongruent_mask = (incong_A_mask + incong_V_mask) / 2

            # compute contrast for current subject: mean(unisensory) - mean(incongruent)
            contrast_mask = mean_unisensory_mask - mean_incongruent_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
            
        # superadditivity contrast: congruent > (incongruent_A + incongruent_V)
        elif contrast_val == "superadditivity_congruent-incongruent":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)
            
            # incongruent_A path
            file_name_incong_A = f"sub-mm{sub}_cond-incongruent_A_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_A = f'{sl_dir}/incongruent_A/MNI152NLin2009cAsym'
            incong_A_path = join(mni_folder_incong_A, file_name_incong_A)
            
            # incongruent_V path  
            file_name_incong_V = f"sub-mm{sub}_cond-incongruent_V_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_V = f'{sl_dir}/incongruent_V/MNI152NLin2009cAsym'
            incong_V_path = join(mni_folder_incong_V, file_name_incong_V)
            
            # load files
            congruent_mask = image_read(congruent_path)
            incong_A_mask = image_read(incong_A_path)
            incong_V_mask = image_read(incong_V_path)
            
            # compute superadditivity: congruent - (incongruent_A + incongruent_V)
            contrast_mask = congruent_mask - (incong_A_mask + incong_V_mask)
            
            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
        
        # voxelwise maximum contrast: congruent - max(incongruent_A, incongruent_V)
        elif contrast_val == "superadditivity_congruent-unisensory":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)
            
            # incongruent_A path
            file_name_A = f"sub-mm{sub}_cond-auditory_searchlight-rsa_space_mni.nii.gz"
            mni_folder_A = f'{sl_dir}/auditory/MNI152NLin2009cAsym'
            A_path = join(mni_folder_A, file_name_A)
            
            # incongruent_V path  
            file_name_V = f"sub-mm{sub}_cond-visual_searchlight-rsa_space_mni.nii.gz"
            mni_folder_V = f'{sl_dir}/visual/MNI152NLin2009cAsym'
            V_path = join(mni_folder_V, file_name_V)
            
            # load files
            congruent_mask = image_read(congruent_path)
            A_mask = image_read(A_path)
            V_mask = image_read(V_path)
            
            # compute superadditivity: congruent - (A + V)
            contrast_mask = congruent_mask - (A_mask + V_mask)
            
            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
        
        # voxelwise maximum contrast: congruent - max(incongruent_A, incongruent_V)
        elif contrast_val == "congruent-incongruent_max":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)
            
            # incongruent_A path
            file_name_incong_A = f"sub-mm{sub}_cond-incongruent_A_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_A = f'{sl_dir}/incongruent_A/MNI152NLin2009cAsym'
            incong_A_path = join(mni_folder_incong_A, file_name_incong_A)
            
            # incongruent_V path  
            file_name_incong_V = f"sub-mm{sub}_cond-incongruent_V_searchlight-rsa_space_mni.nii.gz"
            mni_folder_incong_V = f'{sl_dir}/incongruent_V/MNI152NLin2009cAsym'
            incong_V_path = join(mni_folder_incong_V, file_name_incong_V)
            
            # load files
            congruent_mask = image_read(congruent_path)
            incong_A_mask = image_read(incong_A_path)
            incong_V_mask = image_read(incong_V_path)
            
            # compute voxelwise maximum of incongruent conditions
            max_incongruent_mask = np.maximum(incong_A_mask, incong_V_mask)
            
            # compute contrast: congruent - max(incongruent_A, incongruent_V)
            contrast_mask = congruent_mask - max_incongruent_mask
            
            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
            
        # voxelwise maximum contrast: congruent - max(incongruent_A, incongruent_V)
        elif contrast_val == "congruent-unisensory_max":
            
            # congruent condition path
            file_name_congruent = f"sub-mm{sub}_cond-congruent_searchlight-rsa_space_mni.nii.gz"
            mni_folder_congruent = f'{sl_dir}/congruent/MNI152NLin2009cAsym'
            congruent_path = join(mni_folder_congruent, file_name_congruent)
            
            # incongruent_A path
            file_name_A = f"sub-mm{sub}_cond-auditory_searchlight-rsa_space_mni.nii.gz"
            mni_folder_A = f'{sl_dir}/auditory/MNI152NLin2009cAsym'
            A_path = join(mni_folder_A, file_name_A)
            
            # incongruent_V path  
            file_name_V = f"sub-mm{sub}_cond-visual_searchlight-rsa_space_mni.nii.gz"
            mni_folder_V = f'{sl_dir}/visual/MNI152NLin2009cAsym'
            V_path = join(mni_folder_V, file_name_V)
            
            # load files
            congruent_mask = image_read(congruent_path)
            A_mask = image_read(A_path)
            V_mask = image_read(V_path)
            
            # compute voxelwise maximum of incongruent conditions
            max_unisesory_mask = np.maximum(A_mask, V_mask)
            
            # compute contrast: congruent - max(incongruent_A, incongruent_V)
            contrast_mask = congruent_mask - max_unisesory_mask
            
            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)
            
        else:
        
            # condition 1 path for curr. sub
            file_name_i = f"sub-mm{sub}_cond-{conds[0]}_searchlight-rsa_space_mni.nii.gz"
            mni_folder_cond_i = f'{sl_dir}/{conds[0]}/MNI152NLin2009cAsym'
            cond_i_path = join(mni_folder_cond_i, file_name_i)

            # condition 2 path for curr. sub
            file_name_ii = f"sub-mm{sub}_cond-{conds[1]}_searchlight-rsa_space_mni.nii.gz"
            mni_folder_cond_ii = f'{sl_dir}/{conds[1]}/MNI152NLin2009cAsym'
            cond_ii_path = join(mni_folder_cond_ii, file_name_ii)

            # load files
            cond_i_mask = image_read(cond_i_path)
            cond_ii_mask = image_read(cond_ii_path)

            # compute contrast for current subject
            contrast_mask = cond_i_mask - cond_ii_mask

            # save contrast mni mask
            contrast_file = f"sub-mm{sub}_contrast-{contrast_val}_searchlight-rsa_space-MNI152NLin2009cAsym.nii.gz"
            contrast_path = join(contrast_dir, contrast_file) 
            image_write(contrast_mask, contrast_path)